In [ ]:
import json
import pandas as pd
import requests

# 1. Load archetype JSON and topic keywords mapping
authority_json_path = "karakteristik_arketipe_enriched.json"
topic_kw_path = "daftar_topik_keywords_automerged.csv"

with open(authority_json_path, "r", encoding="utf-8") as f:
    archetypes = json.load(f)

topic_kw_df = pd.read_csv(topic_kw_path)
topic_mapping = {int(r['topic']): r['keywords'].split(', ')[:3] for _, r in topic_kw_df.iterrows()}

# 2. Configure API\ nAPI_KEY = "GROQ_YOUR_API_KEY"
ENDPOINT = "https://api.groq.com/openai/v1/chat/completions"
#API_KEY = "token"

# 3. Build revised prompt: interpret topic meaning, not just list keywords
system_msg = {
    "role": "system",
    "content": (
        "Anda adalah seorang Data Analyst berpengalaman dan pakar perilaku pemain game. "
        "Anda akan menerima data JSON setiap arketipe pemain. Ikuti instruksi berikut untuk masing-masing arketipe:\n"
        "1. arketipe: tulis nama key dari JSON.\n"
        "2. fitur_kuantitatif: ringkas dengan angka aktual, format: '226.58 game dimiliki, 25.61 jam playtime, 2456 achievement'.\n"
        "3. topik_dominan: interpretasikan arti topik berdasar kata kunci mapping berikut: "
        f"{json.dumps(topic_mapping, ensure_ascii=False)}. "
        "Contoh: 'pemain sering membahas aspek visual dan musik dari game'.\n"
        "4. interpretasi: 1-2 kalimat dalam bahasa Indonesia yang mencakup angka aktual, makna topik, dan opini (misalnya 'pemain ini sangat ambisius' jika achievement tinggi).\n"
        "RESPOND ONLY WITH A JSON ARRAY, tanpa teks tambahan."
    )
}

# 4. User message: provide archetypes inline as JSON
user_msg = {"role": "user", "content": json.dumps(archetypes, ensure_ascii=False)}

# 5. Send request
payload = {
    "model": "llama3-8b-8192",
    "messages": [system_msg, user_msg],
    "temperature": 0.5,
    "max_tokens": 1024
}
headers = {"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"}
response = requests.post(ENDPOINT, headers=headers, json=payload)
response.raise_for_status()
raw = response.json()["choices"][0]["message"]["content"].strip()

# 6. Extract JSON array from raw text
start = raw.find('[')
end = raw.rfind(']')
json_text = raw[start:end+1] if start != -1 and end != -1 else raw

# 7. Parse JSON or report raw
try:
    parsed = json.loads(json_text)
except json.JSONDecodeError:
    print("⚠️ Parse JSON gagal. Raw output:\n", raw)
    raise

# 8. Save and print
with open("interpretasi_arketipe.json","w",encoding="utf-8") as f:
    json.dump(parsed, f, ensure_ascii=False, indent=4)

print("✅ JSON interpretasi disimpan di 'interpretasi_arketipe.json'")
print(json.dumps(parsed, ensure_ascii=False, indent=4))


✅ JSON interpretasi disimpan di 'interpretasi_arketipe.json'
[
    {
        "arketype": "archetype_1_weight",
        "fitur_kuantitatif": "226.58 game dimiliki, 25.61 jam playtime, 2456 achievement",
        "topik_dominan": "pemain sering membahas tentang aspek game yang terkait dengan kesehatan dan penyakit",
        "interpretasi": "pemain ini memiliki minat yang kuat pada game yang terkait dengan kesehatan dan penyakit, seperti pinball dan game lainnya."
    },
    {
        "arketype": "archetype_2_weight",
        "fitur_kuantitatif": "231.29 game dimiliki, 35.03 jam playtime, 2607 achievement",
        "topik_dominan": "pemain sering membahas tentang masalah teknis dan bug dalam game",
        "interpretasi": "pemain ini memiliki masalah teknis yang sering terjadi dalam game, seperti crash dan lag."
    },
    {
        "arketype": "archetype_3_weight",
        "fitur_kuantitatif": "247.43 game dimiliki, 29.85 jam playtime, 3560 achievement",
        "topik_dominan": "pemain s